# Module 13: MCP tools inside an agent loop

**MCP** (Model Context Protocol) is a standard way for a program to expose **tools** that any agent can use.

This notebook has two parts:
1. An **MCP server** that offers two tools: `add` and `word_count`.
2. An **agent loop** that asks the LLM, runs the tools it requests through MCP, feeds the results back, and repeats.

Run the cells **in order, one at a time** (`Shift+Enter`).

## Step 0: Imports and LLM config
Same local Ollama setup as the earlier modules.

In [ ]:
import json
import os
import sys
from contextlib import AsyncExitStack

from openai import OpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "qwen2.5:7b"
LLM_OPTIONS = {"reasoning_effort": "none"} if API_KEY == "ollama" else {}
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

MAX_TURNS = 6
SERVER_PATH = os.path.abspath("13_mcp_server.py")  # created in the next cell
print("LLM:", MODEL, "| server file:", SERVER_PATH)

LLM: qwen2.5:7b | server file: c:\Users\yy\Desktop\codes_PhD\agent_ai\tuto\13_mcp_server.py


## Step 1: Write the MCP server

An MCP server is just a program. Here we write it to a file so the agent can start it as a subprocess and talk to it over **stdio**.
The `@mcp.tool()` decorator turns a Python function into a tool. The function's name, docstring and type hints become the tool's description for the LLM.

In [ ]:
%%writefile 13_mcp_server.py
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("demo-tools")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b


@mcp.tool()
def word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())


if __name__ == "__main__":
    mcp.run()  # default transport: stdio

Overwriting 13_mcp_server.py


## Step 2: Start the server and connect a client

The agent starts the server subprocess and opens a session with it.
`AsyncExitStack` keeps the connection open across cells, so you can run the next cells without restarting it.

In [ ]:
stack = AsyncExitStack()
params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
# Jupyter's stderr has no file descriptor, so send the server's stderr to a log file
errlog = stack.enter_context(open("13_mcp_server.log", "w", encoding="utf-8"))
read, write = await stack.enter_async_context(stdio_client(params, errlog=errlog))
session = await stack.enter_async_context(ClientSession(read, write))
await session.initialize()
print("Connected to MCP server.")

Connected to MCP server.


## Step 3: Ask the server which tools it has

`list_tools()` is the MCP call that returns each tool's name, description and input schema.

In [ ]:
mcp_tools = (await session.list_tools()).tools
for t in mcp_tools:
    print(t.name, "->", t.description)

add -> Add two numbers and return the sum.
word_count -> Count the words in a piece of text.


## Step 4: Convert MCP tools into the LLM's tool format

MCP and the OpenAI-style API use similar schemas, but the wrapper differs. We convert each MCP tool into the format the LLM expects.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description or "",
            "parameters": t.input_schema,
        },
    }
    for t in mcp_tools
]
print(json.dumps(tools[0], indent=2))

{
  "type": "function",
  "function": {
    "name": "add",
    "description": "Add two numbers and return the sum.",
    "parameters": {
      "properties": {
        "a": {
          "title": "A",
          "type": "number"
        },
        "b": {
          "title": "B",
          "type": "number"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object",
      "title": "addArguments"
    }
  }
}


## Step 5: The agent loop

The loop:
1. Call the LLM with the full message history and the tool list.
2. If the reply has **no tool calls**, the model is done, so return its answer.
3. If it **has tool calls**, run each one through MCP with `session.call_tool(...)`, append the result as a `tool` message, and go back to step 1.

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Use the tools whenever the task needs counting or arithmetic.
Only give your final answer once you have all the information you need.
Give only numbers that a tool returned. If a tool returns an error, do not guess the value:
call the tool again with corrected arguments, or say that it failed.
Execute a tool only when all of its parameters are already known. If a parameter depends on
another tool's result, call only the tool that produces that result first, wait for its result,
and call the dependent tool in a later turn. Never guess a parameter value.
"""


async def run_agent(task: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]

    for turn in range(MAX_TURNS):
        reply = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            parallel_tool_calls=False,  # one tool call per turn, so each call sees the previous result
            **LLM_OPTIONS
        )
        msg = reply.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        # Show what the model wrote and what it decided to do
        print(f"[turn {turn + 1}] message.content = {msg.content!r}")
        if msg.tool_calls:
            print(f"[turn {turn + 1}] decided to call: {[c.function.name for c in msg.tool_calls]}")
        else:
            print(f"[turn {turn + 1}] no tool call -> treating this as the final answer")

        # No tool requested -> the model is done
        if not msg.tool_calls:
            return msg.content

        # Run each requested tool through MCP and send the result back
        for i, call in enumerate(msg.tool_calls):
            # Only the first tool runs per turn. The model can't see the result of the
            # others yet, so it may guess their arguments. Reply to the skipped calls anyway,
            # because the API requires a reply for every tool call.
            if i > 0:
                print(f"[turn {turn + 1}] skipped {call.function.name} (one tool per turn)")
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": "SKIPPED: only one tool runs per turn. Call it in a later turn if you still need it.",
                })
                continue
            args = json.loads(call.function.arguments)
            result = await session.call_tool(call.function.name, args)
            text = result.content[0].text
            # A failed tool call comes back as a normal result with is_error=True,
            # so label it clearly so the model can't mistake it for a real value
            if result.is_error:
                text = f"TOOL ERROR: {text}. Fix the arguments and call the tool again."
            print(f"[turn {turn + 1}] tool={call.function.name} args={args} -> {text}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": text,
            })

    return "Stopped: reached MAX_TURNS without a final answer."

## Step 6: Run the agent on a task
The task needs both tools: one to count words, one to do arithmetic.

In [ ]:
task = ("How many words are in the sentence "
        "'MCP lets agents use outside tools', and what is that number plus 27?")
answer = await run_agent(task)
print("\nFinal answer:", answer)

[turn 1] message.content = ''
[turn 1] decided to call: ['word_count', 'add']
[turn 1] tool=word_count args={'text': 'MCP lets agents use outside tools'} -> 6
[turn 1] skipped add (one tool per turn)
[turn 2] message.content = ''
[turn 2] decided to call: ['add']
[turn 2] tool=add args={'b': 27, 'a': 6} -> 33.0
[turn 3] message.content = '33'
[turn 3] no tool call -> treating this as the final answer

Final answer: 33


## Step 7: Clean up

When you're done, **restart the kernel** (Kernel → Restart). That closes the connection and stops the MCP server subprocess.
(We don't close it in a cell: the connection was opened in an earlier cell, and closing it from a different cell raises an anyio error.)

## Things to try
- Change the task and watch which tools the model picks.
- Add a third tool to the server (for example `multiply`). The agent will pick it up automatically, because it reads the tool list from the server.
- The model may pass a wrong argument (the small `qwen2.5:7b` model sometimes does). The tool runs correctly, but the answer is only as good as the arguments the model chooses. A stronger model usually makes fewer of these mistakes.